In [61]:
import re
import nltk
import numpy as np
import pandas as pd
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
CSVPATH = 'bbc-text (1).csv' #YOUR PATH
df = pd.read_csv(CSVPATH)

In [63]:
df.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


# Data Preprocessing

In [64]:
class Preprocessor:
  #Remove any punctuation marks or other special characters
  def remove_special_characters(self,strinput):
    pattern = r"[^\w\s_]+|_"
    removed = re.sub(pattern, '',strinput)
    return removed
  #Remove any numbers
  def remove_digits(self,strinput):
    pattern = r"\d+"
    removed = re.sub(pattern, '',strinput)
    return removed
  #Convert all the letters to lower case
  def to_lower(self,strinput):
    return strinput.lower()
  #Remove any common words that do not add much meaning
  def remove_stop_words(self,strinput):
    tokens = nltk.word_tokenize(strinput)
    stopwords = nltk.corpus.stopwords.words("english")
    filtered_tokens = [token for token in tokens if token.lower() not in stopwords]
    return filtered_tokens

In [65]:
preproc = Preprocessor()
df['tokenized'] = (
    df['text'].apply(preproc.remove_special_characters)
    .apply(preproc.remove_digits)
    .apply(preproc.to_lower)
    .apply(preproc.remove_stop_words)
)
df[['text','tokenized']].head()

#train/test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['category'])

# Implement Bag of Words (BoW)

In [66]:
class BagOfWords:
  def __init__(self, data):
    self.data = data
    self.bowvocab = sorted(list(set(data['tokenized'].sum())))
    self.wordindex = {word: index for index, word in enumerate(self.bowvocab)}
  def vector(self, doc):
    vec = np.zeros(len(self.bowvocab))
    for token in doc:
      if token in self.wordindex:
        vec[self.wordindex[token]] += 1
    return vec

In [67]:
bow = BagOfWords(train_df)
train_df['bow'] = train_df['tokenized'].apply(bow.vector)
test_df['bow'] = test_df['tokenized'].apply(bow.vector)

In [68]:
print(len(bow.bowvocab))
train_df[['text','bow']].head()

27446


,text,bow
593,share boost for feud-hit reliance the board of...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
376,iaaf will contest greek decision the internati...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1644,spirit awards hail sideways the comedy sideway...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
876,anglers could face prosecution anglers and f...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1014,howard dismisses tory tax fears michael howard...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


# Implement TF-IDF

In [69]:
class TFIDF:
  def __init__(self, data):
    self.data = data
    self.tfidfvocab = sorted(list(set(data['tokenized'].sum())))
    self.wordindex = {word: index for index, word in enumerate(self.tfidfvocab)}
    self.wordcount = len(self.tfidfvocab)
    self.doccount = len(data)
  def tffunc(self, doc):
    vec = np.zeros(self.wordcount)
    if len(doc) == 0:
      return vec
    for token in doc:
      if token in self.wordindex:
        vec[self.wordindex[token]] += 1
    return vec/len(doc)
  def idffunc(self):
    n_t = np.zeros(self.wordcount)
    for doc in self.data['tokenized']:
      for token in set(doc):
        if token in self.wordindex:
          n_t[self.wordindex[token]] += 1
    n_t = np.where(n_t == 0, 1e-7, n_t)
    self.idf = np.log(self.doccount/n_t)
    return self.idf
  def tfidffunc(self, tf):
    return tf*self.idf

In [70]:
tfidf = TFIDF(train_df)
IDF = tfidf.idffunc()
print(IDF)

[7.48436864 5.69260917 5.87493073 ... 5.18178355 7.48436864 6.79122146]


In [71]:
train_df['tf'] = train_df['tokenized'].apply(tfidf.tffunc)
test_df['tf'] = test_df['tokenized'].apply(tfidf.tffunc)
train_df['tf'].head()

,tf
593,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
376,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1644,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
876,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1014,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [72]:
train_df['tfidf'] = train_df['tf'].apply(tfidf.tfidffunc)
test_df['tfidf'] = test_df['tf'].apply(tfidf.tfidffunc)
train_df['tfidf'].head()

,tfidf
593,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
376,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1644,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
876,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1014,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


# Analysis

In [73]:
categories = train_df['category'].unique()
print(categories)

['business' 'sport' 'entertainment' 'politics' 'tech']


In [79]:
for category in categories:
    category_df = train_df[train_df['category'] == category]
    avg_tfidf = np.mean(np.array(category_df['tfidf'].tolist()),axis=0)
    top_10_words = avg_tfidf.argsort()[-10:][::-1]
    print(f"Top 10 words for {category}:")
    words = []
    for index in top_10_words:
        words.append(tfidf.tfidfvocab[index])
    print(" ".join(words))
    print()

Top 10 words for business:
bn bank growth sales shares economy oil company us market

Top 10 words for sport:
england cup game match club win chelsea injury season coach

Top 10 words for entertainment:
film best show music band festival awards award album actor

Top 10 words for politics:
labour mr blair election party government brown howard minister lord

Top 10 words for tech:
software mobile users people technology microsoft digital broadband computer phone



In [75]:
avg_tf = np.mean(np.array(train_df['tf'].tolist()),axis=0)
word_scores = pd.DataFrame({'word': tfidf.tfidfvocab,'TF': avg_tf,'IDF': IDF})
high_tf_low_idf = word_scores[(word_scores['TF'] >= word_scores['TF'].quantile(0.75)) & (word_scores['IDF'] <= word_scores['IDF'].quantile(0.25))].sort_values('TF', ascending=False)
low_tf_high_idf = word_scores[(word_scores['TF'] <= word_scores['TF'].quantile(0.25)) & (word_scores['IDF'] >= word_scores['IDF'].quantile(0.75))].sort_values('IDF', ascending=False)
print("High TF + Low IDF:")
print(high_tf_low_idf.head(20))
print("\nLow TF + High IDF:")
print(low_tf_high_idf.head(20))

High TF + Low IDF:
             word        TF       IDF
21128        said  0.015749  0.154619
15901          mr  0.005565  1.041828
27152       would  0.005170  0.649184
757          also  0.004692  0.554852
27294        year  0.004485  0.776285
25975          us  0.004281  0.995164
16339         new  0.004088  0.813602
17860      people  0.003593  1.032320
16960         one  0.003492  0.778730
13671        last  0.003464  0.865630
5433        could  0.002944  0.937583
9104        first  0.002826  1.067636
25397         two  0.002621  1.005859
27123       world  0.002611  1.198371
24739        time  0.002318  1.092452
2684           bn  0.002276  1.893382
25465          uk  0.002252  1.441736
10361  government  0.002241  1.541569
27305       years  0.002135  1.205847
8997         film  0.002044  2.232095

Low TF + High IDF:
                  word            TF       IDF
27442           zubair  2.625223e-06  7.484369
0                   aa  1.692162e-06  7.484369
7                aamir

# Category Classification Using Logistic Regression

In [76]:
y_train = train_df['category']
y_test = test_df['category']

X_train_bow = np.array(train_df['bow'].tolist())
X_test_bow = np.array(test_df['bow'].tolist())

bow_classifier = LogisticRegression(max_iter=1000,random_state=42)
bow_classifier.fit(X_train_bow, y_train)
bow_pred = bow_classifier.predict(X_test_bow)

print("BoW Accuracy:", accuracy_score(y_test, bow_pred))
print("Classification Report:")
print(classification_report(y_test, bow_pred))

BoW Accuracy: 0.9797752808988764
Classification Report:
               precision    recall  f1-score   support

     business       0.98      0.97      0.98       102
entertainment       0.95      0.99      0.97        77
     politics       0.98      0.99      0.98        84
        sport       1.00      1.00      1.00       102
         tech       0.99      0.95      0.97        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445



In [77]:
X_train_tfidf = np.array(train_df['tfidf'].tolist())
y_train_tfidf = train_df['category']

X_test_tfidf = np.array(test_df['tfidf'].tolist())
y_test_tfidf = test_df['category']

tfidf_classifier = LogisticRegression(max_iter=1000, random_state=42)
tfidf_classifier.fit(X_train_tfidf, y_train)
tfidf_pred = tfidf_classifier.predict(X_test_tfidf)

print(f"TFIDF Accuracy: {accuracy_score(y_test, tfidf_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, tfidf_pred))

TFIDF Accuracy: 0.9685

Classification Report:
               precision    recall  f1-score   support

     business       0.92      0.98      0.95       102
entertainment       1.00      0.97      0.99        77
     politics       0.98      0.95      0.96        84
        sport       0.97      1.00      0.99       102
         tech       1.00      0.93      0.96        80

     accuracy                           0.97       445
    macro avg       0.97      0.97      0.97       445
 weighted avg       0.97      0.97      0.97       445



In [80]:
train_df.to_csv("train_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)